# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a tutorial for loading and exploring a dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

The dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples, including various fields such as accession, coverage percentage, peptide counts, and molecular weight.

### Dataset Source
The dataset source Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, columns, and their IDs.

In [ ]:
# List all RecordSets and their details, referencing by '@id'

print("Available RecordSets:")
for record_set in metadata.record_sets:
    print(f"\nRecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    if 'description' in record_set:
        print(f"  Description: {record_set['description']}")
    # List fields in this record set
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            print(f"    - Field @id: {field['@id']}")
            print(f"      Name: {field.get('name', '')}")
            print(f"      Data Type: {field.get('dataType', '')}")
            print(f"      Source Column: {field.get('column', [{}])[0].get('@id', '') if field.get('column') else ''}")


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and Field `@id`s from the overview above.

In [ ]:
# List all available RecordSet ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print("RecordSet @ids available:", record_set_ids)

# Select the first RecordSet as example (you can choose others from above list)
selected_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"Using RecordSet: {selected_record_set_id}")

# Load all records from each RecordSet into DataFrames, referenced by @id
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

if selected_record_set_id is not None:
    print(f"Fields in '{selected_record_set_id}':")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Here we:
 - Filter for records with a numeric field greater than a threshold.
 - Normalize that field.
 - Group by a categorical field and compute means.


In [ ]:
rs_id = selected_record_set_id
df = dataframes[rs_id]

# Heuristically pick a numeric field (could also inspect the schema programmatically)
possible_numeric_fields = [col for col in df.columns if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col])]
if not possible_numeric_fields:
    # Try to coerce to numeric
    for col in df.columns:
        coerced = pd.to_numeric(df[col], errors='coerce')
        if coerced.notnull().sum() > 0:
            df[col+'_numeric'] = coerced
            possible_numeric_fields.append(col+'_numeric')
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else None

if numeric_field:
    print(f"Analyzing numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (top 5):")
    display(filtered_df.head())

    # Normalize
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (top 5):")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Heuristically pick a categorical/grouping field
    possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped averages by {group_field} (top 5):")
        display(grouped_df.head())
    else:
        print("No categorical field available for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to access, load, and analyze the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library. We surveyed the available record sets, fields, and extracted tabular records for exploration and visualization—all referencing entities by their `@id` in alignment with FAIR principles.

Key findings depend on the data content: the sample workflow shows how to filter, normalize, group, and plot fields in any Croissant-structured dataset. For domain-specific conclusions, repeat the process focusing on scientific questions of interest (e.g., comparison of protein abundances by accession or experimental condition).

For more, check the [Croissant standard documentation](https://mlcommons.org/croissant/) or the [mlcroissant Python API](https://mlcommons.org/croissant/implementations/python/).
